# TCGA-BRCA Biospecimen Identifier Crosswalk Review

This notebook is review-only. It loads the latest saved biospecimen identifier crosswalk outputs from disk, checks the run-level validation state, and writes review tables for human source audit.


## Load the latest saved biospecimen identifier crosswalk run


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'variables'
    / 'tcga_brca_biospecimen_identifier_crosswalk_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest biospecimen identifier crosswalk pointer not found: {latest_pointer_path}. Run the crosswalk script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
inventory_path = repo_root / latest_pointer['biospecimen_identifier_inventory_tsv']
crosswalk_path = repo_root / latest_pointer['biospecimen_identifier_crosswalk_tsv']
summary_path = repo_root / latest_pointer['biospecimen_identifier_crosswalk_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_pointer]))


,updated_at_utc,crosswalk_run_id,parse_run_id,source_run_id,crosswalk_run_directory,biospecimen_identifier_inventory_tsv,biospecimen_identifier_crosswalk_tsv,biospecimen_identifier_crosswalk_summary_tsv,run_log_json,biospecimen_biotab_latest_json,field_count
0,2026-04-13T07:36:40Z,20260413T073640Z,20260413T070359Z,20260412T000556Z,01-data/audit/tcga-brca/variables/biospecimen_...,01-data/audit/tcga-brca/variables/biospecimen_...,01-data/audit/tcga-brca/variables/biospecimen_...,01-data/audit/tcga-brca/variables/biospecimen_...,01-data/audit/tcga-brca/variables/biospecimen_...,01-data/audit/tcga-brca/variables/tcga_brca_bi...,47


## Load saved biospecimen identifier crosswalk artifacts


In [2]:
inventory_df = pd.read_csv(inventory_path, sep='\t')
crosswalk_df = pd.read_csv(crosswalk_path, sep='\t')
summary_df = pd.read_csv(summary_path, sep='\t')
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

family_order = [
    'patient_identifier_like',
    'sample_identifier_like',
    'portion_identifier_like',
    'analyte_identifier_like',
    'slide_identifier_like',
    'aliquot_identifier_like',
    'multi_level_or_unclear_identifier_like',
]
role_order = [
    'primary_link_candidate',
    'overlapping_link_candidate',
    'secondary_link_candidate',
    'side_table_candidate',
    'ambiguous_candidate',
]
candidate_source_order = [
    'field_name_level_and_identifier_token_match',
    'field_name_level_token_match_only',
    'field_name_identifier_token_match_only',
    'field_name_multi_level_token_match',
]

inventory_df['probable_identifier_level'] = inventory_df['probable_identifier_level'].fillna('')
inventory_df['identifier_token_family'] = inventory_df['identifier_token_family'].fillna('')
inventory_df['candidate_source'] = pd.Categorical(
    inventory_df['candidate_source'], categories=candidate_source_order, ordered=True
)
inventory_df['manual_review_priority'] = pd.Categorical(
    inventory_df['manual_review_priority'], categories=['high', 'medium', 'low'], ordered=True
)
crosswalk_df['identifier_linkage_family'] = pd.Categorical(
    crosswalk_df['identifier_linkage_family'], categories=family_order, ordered=True
)
crosswalk_df['crosswalk_role'] = pd.Categorical(
    crosswalk_df['crosswalk_role'], categories=role_order, ordered=True
)
crosswalk_df['candidate_source'] = pd.Categorical(
    crosswalk_df['candidate_source'], categories=candidate_source_order, ordered=True
)
crosswalk_df['manual_review_priority'] = pd.Categorical(
    crosswalk_df['manual_review_priority'], categories=['high', 'medium', 'low'], ordered=True
)
summary_df['identifier_linkage_family'] = pd.Categorical(
    summary_df['identifier_linkage_family'], categories=family_order, ordered=True
)

print(f'Identifier inventory TSV: {inventory_path}')
print(f'Identifier crosswalk TSV: {crosswalk_path}')
print(f'Identifier crosswalk summary TSV: {summary_path}')
print(f'Run log: {run_log_path}')
print(f"Crosswalk run ID: {latest_pointer['crosswalk_run_id']}")
print(f"Parse run ID: {latest_pointer['parse_run_id']}")
print(f"Source run ID: {latest_pointer['source_run_id']}")
display(pd.DataFrame([run_log['validation']]))
display(summary_df.sort_values('identifier_linkage_family').reset_index(drop=True))
display(crosswalk_df.head(20))


Identifier inventory TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\biospecimen_identifier_crosswalk_runs\20260413T073640Z\biospecimen_identifier_inventory.tsv
Identifier crosswalk TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\biospecimen_identifier_crosswalk_runs\20260413T073640Z\biospecimen_identifier_crosswalk.tsv
Identifier crosswalk summary TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\biospecimen_identifier_crosswalk_runs\20260413T073640Z\biospecimen_identifier_crosswalk_summary.tsv
Run log: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\biospecimen_identifier_crosswalk_runs\20260413T073640Z\run_log.json
Crosswalk run ID: 20260413T073640Z
Parse run ID: 20260413T070359Z
Source run ID: 20260412T000556Z


,passed,biospecimen_parse_latest_pointer_found,biospecimen_parse_run_log_completed,required_central_tables_found,optional_side_tables_loaded,identifier_like_field_count_positive,inventory_row_count_positive,crosswalk_row_count_positive,summary_row_count_positive,inventory_candidate_sources_valid,crosswalk_roles_and_rules_valid,summary_family_counts_match_crosswalk,no_prior_run_overwrite,latest_pointer_written_after_success_only
0,True,True,True,True,"[biospecimen_diagnostic_slides, biospecimen_pr...",True,True,True,True,True,True,True,True,True


,crosswalk_run_id,parse_run_id,source_run_id,identifier_linkage_family,field_count,tables_present_json,primary_candidates_json,secondary_candidates_json,overlapping_candidates_json,ambiguous_candidates_json,side_table_candidates_json,notes_placeholder
0,20260413T073640Z,20260413T070359Z,20260412T000556Z,patient_identifier_like,9,"[""biospecimen_aliquot"", ""biospecimen_analyte"",...","[""biospecimen_diagnostic_slides.bcr_patient_ba...",[],"[""biospecimen_aliquot.bcr_patient_uuid"", ""bios...",[],"[""biospecimen_diagnostic_slides.bcr_patient_uu...",[fill in during biospecimen identifier crosswa...
1,20260413T073640Z,20260413T070359Z,20260412T000556Z,sample_identifier_like,14,"[""biospecimen_aliquot"", ""biospecimen_analyte"",...","[""biospecimen_sample.bcr_sample_barcode"", ""bio...","[""biospecimen_sample.days_to_sample_procuremen...","[""biospecimen_aliquot.bcr_sample_barcode"", ""bi...",[],"[""biospecimen_diagnostic_slides.bcr_sample_bar...",[fill in during biospecimen identifier crosswa...
2,20260413T073640Z,20260413T070359Z,20260412T000556Z,portion_identifier_like,11,"[""biospecimen_analyte"", ""biospecimen_portion"",...","[""biospecimen_portion.bcr_portion_barcode"", ""b...","[""biospecimen_analyte.subportion_sequence"", ""b...",[],[],"[""biospecimen_shipment_portion.bcr_shipment_po...",[fill in during biospecimen identifier crosswa...
3,20260413T073640Z,20260413T070359Z,20260412T000556Z,analyte_identifier_like,5,"[""biospecimen_analyte"", ""biospecimen_protocol""]","[""biospecimen_analyte.bcr_analyte_barcode"", ""b...","[""biospecimen_analyte.analyte_type"", ""biospeci...",[],[],"[""biospecimen_protocol.bcr_analyte_barcode""]",[fill in during biospecimen identifier crosswa...
4,20260413T073640Z,20260413T070359Z,20260412T000556Z,slide_identifier_like,3,"[""biospecimen_diagnostic_slides"", ""biospecimen...","[""biospecimen_slide.bcr_slide_barcode"", ""biosp...",[],[],[],"[""biospecimen_diagnostic_slides.ffpe_slide_uuid""]",[fill in during biospecimen identifier crosswa...
5,20260413T073640Z,20260413T070359Z,20260412T000556Z,aliquot_identifier_like,2,"[""biospecimen_aliquot""]","[""biospecimen_aliquot.bcr_aliquot_barcode"", ""b...",[],[],[],[],[fill in during biospecimen identifier crosswa...
6,20260413T073640Z,20260413T070359Z,20260412T000556Z,multi_level_or_unclear_identifier_like,3,"[""biospecimen_aliquot"", ""biospecimen_sample"", ...",[],[],[],"[""biospecimen_aliquot.biospecimen_barcode_bott...",[],[fill in during biospecimen identifier crosswa...


,crosswalk_run_id,parse_run_id,source_run_id,identifier_linkage_family,table_name,field_name,source_position,probable_identifier_level,identifier_token_family,candidate_source,...,manual_review_priority,notes_placeholder,matched_identifier_tokens_json,alternate_column_name,cde_id_raw,paired_field_name,pattern_rule,pattern_comparable_row_count,pattern_pass_fraction,pattern_nonmatch_examples_json
0,20260413T073640Z,20260413T070359Z,20260412T000556Z,patient_identifier_like,biospecimen_sample,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,...,low,[fill in during biospecimen identifier crosswa...,"[""patient"", ""uuid""]",NaN,NaN,bcr_sample_barcode,same_row_copresence_only,2302,1.000000,[]
1,20260413T073640Z,20260413T070359Z,20260412T000556Z,patient_identifier_like,biospecimen_diagnostic_slides,bcr_patient_barcode,2,patient,barcode,field_name_level_and_identifier_token_match,...,medium,[fill in during biospecimen identifier crosswa...,"[""patient"", ""barcode""]",NaN,CDE_ID:2673794,bcr_patient_uuid,exact_pair_one_to_one_same_table,1142,1.000000,[]
2,20260413T073640Z,20260413T070359Z,20260412T000556Z,patient_identifier_like,biospecimen_portion,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,...,medium,[fill in during biospecimen identifier crosswa...,"[""patient"", ""uuid""]",NaN,NaN,bcr_sample_barcode,same_row_copresence_only,2310,1.000000,[]
3,20260413T073640Z,20260413T070359Z,20260412T000556Z,patient_identifier_like,biospecimen_analyte,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,...,medium,[fill in during biospecimen identifier crosswa...,"[""patient"", ""uuid""]",NaN,NaN,bcr_sample_barcode,same_row_copresence_only,6315,1.000000,[]
4,20260413T073640Z,20260413T070359Z,20260412T000556Z,patient_identifier_like,biospecimen_slide,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,...,medium,[fill in during biospecimen identifier crosswa...,"[""patient"", ""uuid""]",NaN,NaN,bcr_sample_barcode,same_row_copresence_only,1995,1.000000,[]
5,20260413T073640Z,20260413T070359Z,20260412T000556Z,patient_identifier_like,biospecimen_aliquot,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,...,medium,[fill in during biospecimen identifier crosswa...,"[""patient"", ""uuid""]",NaN,NaN,bcr_sample_barcode,same_row_copresence_only,14538,1.000000,[]
6,20260413T073640Z,20260413T070359Z,20260412T000556Z,patient_identifier_like,biospecimen_protocol,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,...,medium,[fill in during biospecimen identifier crosswa...,"[""patient"", ""uuid""]",NaN,NaN,bcr_sample_barcode,same_row_copresence_only,6315,1.000000,[]
7,20260413T073640Z,20260413T070359Z,20260412T000556Z,patient_identifier_like,biospecimen_shipment_portion,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,...,medium,[fill in during biospecimen identifier crosswa...,"[""patient"", ""uuid""]",NaN,NaN,bcr_sample_barcode,same_row_copresence_only,1064,1.000000,[]
8,20260413T073640Z,20260413T070359Z,20260412T000556Z,patient_identifier_like,biospecimen_diagnostic_slides,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,...,medium,[fill in during biospecimen identifier crosswa...,"[""patient"", ""uuid""]",NaN,NaN,bcr_patient_barcode,exact_pair_one_to_one_same_table,1142,1.000000,[]
9,20260413T073640Z,20260413T070359Z,20260412T000556Z,sample_identifier_like,biospecimen_sample,bcr_sample_barcode,2,sample,barcode,field_name_level_and_identifier_token_match,...,low,[fill in during biospecimen identifier crosswa...,"[""sample"", ""barcode""]",NaN,NaN,bcr_sample_uuid,exact_pair_one_to_one_same_table,2302,1.000000,[]


## Save identifier inventory review table


In [3]:
inventory_review_columns = [
    'table_name',
    'field_name',
    'source_position',
    'probable_identifier_level',
    'identifier_token_family',
    'candidate_source',
    'missing_like_fraction',
    'non_missing_count',
    'distinct_non_missing_count',
    'matched_identifier_tokens_json',
    'alternate_column_name',
    'cde_id_raw',
    'manual_review_priority',
    'example_values_small_sample',
    'notes_placeholder',
]

inventory_review_df = (
    inventory_df.loc[:, inventory_review_columns]
    .sort_values(
        ['table_name', 'candidate_source', 'missing_like_fraction', 'non_missing_count', 'source_position'],
        ascending=[True, True, True, False, True],
    )
    .reset_index(drop=True)
)
inventory_review_path = results_root / '42_biospecimen_identifier_inventory.tsv'
inventory_review_df.to_csv(inventory_review_path, sep='\t', index=False)

print(f'Saved: {inventory_review_path}')
display(inventory_review_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\42_biospecimen_identifier_inventory.tsv


,table_name,field_name,source_position,probable_identifier_level,identifier_token_family,candidate_source,missing_like_fraction,non_missing_count,distinct_non_missing_count,matched_identifier_tokens_json,alternate_column_name,cde_id_raw,manual_review_priority,example_values_small_sample,notes_placeholder
0,biospecimen_aliquot,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,0.000000,14538,1101,"[""patient"", ""uuid""]",NaN,NaN,low,"[""eda6d2d5-4199-4f76-a45b-1d0401b4e54c"", ""4510...",[fill in during biospecimen identifier crosswa...
1,biospecimen_aliquot,bcr_sample_barcode,2,sample,barcode,field_name_level_and_identifier_token_match,0.000000,14538,2300,"[""sample"", ""barcode""]",NaN,NaN,low,"[""TCGA-AR-A1AR-01A"", ""TCGA-AR-A1AR-10A"", ""TCGA...",[fill in during biospecimen identifier crosswa...
2,biospecimen_aliquot,bcr_aliquot_barcode,3,aliquot,barcode,field_name_level_and_identifier_token_match,0.000000,14538,14538,"[""aliquot"", ""barcode""]",NaN,NaN,low,"[""TCGA-AR-A1AR-01A-31D-A133-02"", ""TCGA-AR-A1AR...",[fill in during biospecimen identifier crosswa...
3,biospecimen_aliquot,bcr_aliquot_uuid,4,aliquot,uuid,field_name_level_and_identifier_token_match,0.000000,14538,14538,"[""aliquot"", ""uuid""]",NaN,NaN,low,"[""05c45162-6c94-4a15-accc-b6239451064c"", ""c797...",[fill in during biospecimen identifier crosswa...
4,biospecimen_aliquot,biospecimen_barcode_bottom,5,multi_level_or_unclear,barcode,field_name_identifier_token_match_only,0.000000,14538,14537,"[""barcode""]",NaN,NaN,high,"[""0099016644"", ""0108477580"", ""0108477484"", ""01...",[fill in during biospecimen identifier crosswa...
5,biospecimen_analyte,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,0.000000,6315,1101,"[""patient"", ""uuid""]",NaN,NaN,low,"[""eda6d2d5-4199-4f76-a45b-1d0401b4e54c"", ""4510...",[fill in during biospecimen identifier crosswa...
6,biospecimen_analyte,bcr_sample_barcode,2,sample,barcode,field_name_level_and_identifier_token_match,0.000000,6315,2300,"[""sample"", ""barcode""]",NaN,NaN,low,"[""TCGA-AR-A1AR-01A"", ""TCGA-AR-A1AR-10A"", ""TCGA...",[fill in during biospecimen identifier crosswa...
7,biospecimen_analyte,bcr_analyte_barcode,3,analyte,barcode,field_name_level_and_identifier_token_match,0.000000,6315,6315,"[""analyte"", ""barcode""]",NaN,NaN,low,"[""TCGA-AR-A1AR-01A-31D"", ""TCGA-AR-A1AR-01A-31R...",[fill in during biospecimen identifier crosswa...
8,biospecimen_analyte,bcr_analyte_uuid,4,analyte,uuid,field_name_level_and_identifier_token_match,0.000000,6315,6315,"[""analyte"", ""uuid""]",NaN,NaN,low,"[""3febc6c8-85ae-4d38-ba55-c959959846db"", ""5f5b...",[fill in during biospecimen identifier crosswa...
9,biospecimen_analyte,analyte_type,6,analyte,level_only,field_name_level_token_match_only,0.000000,6315,4,"[""analyte""]",NaN,NaN,medium,"[""DNA"", ""RNA"", ""Repli-G (Qiagen) DNA"", ""Repli-...",[fill in during biospecimen identifier crosswa...


## Save crosswalk review tables by family and role


In [4]:
crosswalk_review_columns = [
    'identifier_linkage_family',
    'crosswalk_role',
    'table_name',
    'field_name',
    'source_position',
    'probable_identifier_level',
    'identifier_token_family',
    'candidate_source',
    'missing_like_fraction',
    'non_missing_count',
    'distinct_non_missing_count',
    'paired_field_name',
    'pattern_rule',
    'pattern_comparable_row_count',
    'pattern_pass_fraction',
    'alternate_column_name',
    'cde_id_raw',
    'manual_review_priority',
    'example_values_small_sample',
    'pattern_nonmatch_examples_json',
    'notes_placeholder',
]

crosswalk_by_family_df = (
    crosswalk_df.loc[:, crosswalk_review_columns]
    .sort_values(
        [
            'identifier_linkage_family',
            'crosswalk_role',
            'table_name',
            'missing_like_fraction',
            'non_missing_count',
            'source_position',
        ],
        ascending=[True, True, True, True, False, True],
    )
    .reset_index(drop=True)
)
crosswalk_by_family_path = results_root / '43_biospecimen_identifier_crosswalk_by_family.tsv'
crosswalk_by_family_df.to_csv(crosswalk_by_family_path, sep='\t', index=False)

primary_candidates_df = (
    crosswalk_by_family_df.loc[crosswalk_by_family_df['crosswalk_role'] == 'primary_link_candidate']
    .reset_index(drop=True)
)
primary_candidates_path = results_root / '44_biospecimen_identifier_primary_candidates.tsv'
primary_candidates_df.to_csv(primary_candidates_path, sep='\t', index=False)

overlapping_candidates_df = (
    crosswalk_by_family_df.loc[crosswalk_by_family_df['crosswalk_role'] == 'overlapping_link_candidate']
    .reset_index(drop=True)
)
overlapping_candidates_path = results_root / '45_biospecimen_identifier_overlapping_candidates.tsv'
overlapping_candidates_df.to_csv(overlapping_candidates_path, sep='\t', index=False)

ambiguous_candidates_df = (
    crosswalk_by_family_df.loc[crosswalk_by_family_df['crosswalk_role'] == 'ambiguous_candidate']
    .reset_index(drop=True)
)
ambiguous_candidates_path = results_root / '46_biospecimen_identifier_ambiguous_candidates.tsv'
ambiguous_candidates_df.to_csv(ambiguous_candidates_path, sep='\t', index=False)

print(f'Saved: {crosswalk_by_family_path}')
print(f'Saved: {primary_candidates_path}')
print(f'Saved: {overlapping_candidates_path}')
print(f'Saved: {ambiguous_candidates_path}')
display(primary_candidates_df)
display(overlapping_candidates_df)
display(ambiguous_candidates_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\43_biospecimen_identifier_crosswalk_by_family.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\44_biospecimen_identifier_primary_candidates.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\45_biospecimen_identifier_overlapping_candidates.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\46_biospecimen_identifier_ambiguous_candidates.tsv


,identifier_linkage_family,crosswalk_role,table_name,field_name,source_position,probable_identifier_level,identifier_token_family,candidate_source,missing_like_fraction,non_missing_count,...,paired_field_name,pattern_rule,pattern_comparable_row_count,pattern_pass_fraction,alternate_column_name,cde_id_raw,manual_review_priority,example_values_small_sample,pattern_nonmatch_examples_json,notes_placeholder
0,patient_identifier_like,primary_link_candidate,biospecimen_diagnostic_slides,bcr_patient_barcode,2,patient,barcode,field_name_level_and_identifier_token_match,0.0,1142,...,bcr_patient_uuid,exact_pair_one_to_one_same_table,1142,1.0,NaN,CDE_ID:2673794,medium,"[""TCGA-AR-A1AR"", ""TCGA-BH-A1EO"", ""TCGA-BH-A1ES...",[],[fill in during biospecimen identifier crosswa...
1,patient_identifier_like,primary_link_candidate,biospecimen_sample,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,0.0,2302,...,bcr_sample_barcode,same_row_copresence_only,2302,1.0,NaN,NaN,low,"[""eda6d2d5-4199-4f76-a45b-1d0401b4e54c"", ""4510...",[],[fill in during biospecimen identifier crosswa...
2,sample_identifier_like,primary_link_candidate,biospecimen_sample,bcr_sample_barcode,2,sample,barcode,field_name_level_and_identifier_token_match,0.0,2302,...,bcr_sample_uuid,exact_pair_one_to_one_same_table,2302,1.0,NaN,NaN,low,"[""TCGA-AR-A1AR-01A"", ""TCGA-AR-A1AR-10A"", ""TCGA...",[],[fill in during biospecimen identifier crosswa...
3,sample_identifier_like,primary_link_candidate,biospecimen_sample,bcr_sample_uuid,3,sample,uuid,field_name_level_and_identifier_token_match,0.0,2302,...,bcr_sample_barcode,exact_pair_one_to_one_same_table,2302,1.0,NaN,NaN,low,"[""5fa9998b-deff-493e-8a8e-dc2422192a48"", ""c1e5...",[],[fill in during biospecimen identifier crosswa...
4,portion_identifier_like,primary_link_candidate,biospecimen_portion,bcr_portion_barcode,3,portion,barcode,field_name_level_and_identifier_token_match,0.0,2310,...,bcr_portion_uuid,exact_pair_one_to_one_same_table,2310,1.0,NaN,NaN,low,"[""TCGA-AR-A1AR-01A-31"", ""TCGA-AR-A1AR-10A-01"",...",[],[fill in during biospecimen identifier crosswa...
5,portion_identifier_like,primary_link_candidate,biospecimen_portion,bcr_portion_uuid,4,portion,uuid,field_name_level_and_identifier_token_match,0.0,2310,...,bcr_portion_barcode,exact_pair_one_to_one_same_table,2310,1.0,NaN,NaN,low,"[""40407260-e805-4c2e-b2a7-13862bc5e494"", ""5b2a...",[],[fill in during biospecimen identifier crosswa...
6,analyte_identifier_like,primary_link_candidate,biospecimen_analyte,bcr_analyte_barcode,3,analyte,barcode,field_name_level_and_identifier_token_match,0.0,6315,...,bcr_analyte_uuid,exact_pair_one_to_one_same_table,6315,1.0,NaN,NaN,low,"[""TCGA-AR-A1AR-01A-31D"", ""TCGA-AR-A1AR-01A-31R...",[],[fill in during biospecimen identifier crosswa...
7,analyte_identifier_like,primary_link_candidate,biospecimen_analyte,bcr_analyte_uuid,4,analyte,uuid,field_name_level_and_identifier_token_match,0.0,6315,...,bcr_analyte_barcode,exact_pair_one_to_one_same_table,6315,1.0,NaN,NaN,low,"[""3febc6c8-85ae-4d38-ba55-c959959846db"", ""5f5b...",[],[fill in during biospecimen identifier crosswa...
8,slide_identifier_like,primary_link_candidate,biospecimen_slide,bcr_slide_barcode,3,slide,barcode,field_name_level_and_identifier_token_match,0.0,1995,...,bcr_slide_uuid,exact_pair_one_to_one_same_table,1995,1.0,NaN,NaN,low,"[""TCGA-AR-A1AR-01A-03-TSC"", ""TCGA-BH-A1EO-01A-...",[],[fill in during biospecimen identifier crosswa...
9,slide_identifier_like,primary_link_candidate,biospecimen_slide,bcr_slide_uuid,4,slide,uuid,field_name_level_and_identifier_token_match,0.0,1995,...,bcr_slide_barcode,exact_pair_one_to_one_same_table,1995,1.0,NaN,NaN,low,"[""3013e9be-aa3e-4986-990c-559982f00e36"", ""f976...",[],[fill in during biospecimen identifier crosswa...


,identifier_linkage_family,crosswalk_role,table_name,field_name,source_position,probable_identifier_level,identifier_token_family,candidate_source,missing_like_fraction,non_missing_count,...,paired_field_name,pattern_rule,pattern_comparable_row_count,pattern_pass_fraction,alternate_column_name,cde_id_raw,manual_review_priority,example_values_small_sample,pattern_nonmatch_examples_json,notes_placeholder
0,patient_identifier_like,overlapping_link_candidate,biospecimen_aliquot,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,0.0,14538,...,bcr_sample_barcode,same_row_copresence_only,14538,1.000000,NaN,NaN,medium,"[""eda6d2d5-4199-4f76-a45b-1d0401b4e54c"", ""4510...",[],[fill in during biospecimen identifier crosswa...
1,patient_identifier_like,overlapping_link_candidate,biospecimen_analyte,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,0.0,6315,...,bcr_sample_barcode,same_row_copresence_only,6315,1.000000,NaN,NaN,medium,"[""eda6d2d5-4199-4f76-a45b-1d0401b4e54c"", ""4510...",[],[fill in during biospecimen identifier crosswa...
2,patient_identifier_like,overlapping_link_candidate,biospecimen_portion,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,0.0,2310,...,bcr_sample_barcode,same_row_copresence_only,2310,1.000000,NaN,NaN,medium,"[""eda6d2d5-4199-4f76-a45b-1d0401b4e54c"", ""4510...",[],[fill in during biospecimen identifier crosswa...
3,patient_identifier_like,overlapping_link_candidate,biospecimen_slide,bcr_patient_uuid,1,patient,uuid,field_name_level_and_identifier_token_match,0.0,1995,...,bcr_sample_barcode,same_row_copresence_only,1995,1.000000,NaN,NaN,medium,"[""eda6d2d5-4199-4f76-a45b-1d0401b4e54c"", ""4510...",[],[fill in during biospecimen identifier crosswa...
4,sample_identifier_like,overlapping_link_candidate,biospecimen_aliquot,bcr_sample_barcode,2,sample,barcode,field_name_level_and_identifier_token_match,0.0,14538,...,bcr_aliquot_barcode,barcode_prefix_same_row,14538,0.999587,NaN,NaN,medium,"[""TCGA-AR-A1AR-01A"", ""TCGA-AR-A1AR-10A"", ""TCGA...","[{""prefix_value"": ""TCGA-E2-A2P5-10A"", ""child_v...",[fill in during biospecimen identifier crosswa...
5,sample_identifier_like,overlapping_link_candidate,biospecimen_analyte,bcr_sample_barcode,2,sample,barcode,field_name_level_and_identifier_token_match,0.0,6315,...,bcr_analyte_barcode,barcode_prefix_same_row,6315,0.999367,NaN,NaN,medium,"[""TCGA-AR-A1AR-01A"", ""TCGA-AR-A1AR-10A"", ""TCGA...","[{""prefix_value"": ""TCGA-E2-A2P5-10A"", ""child_v...",[fill in during biospecimen identifier crosswa...
6,sample_identifier_like,overlapping_link_candidate,biospecimen_portion,bcr_sample_barcode,2,sample,barcode,field_name_level_and_identifier_token_match,0.0,2310,...,bcr_portion_barcode,barcode_prefix_same_row,2310,1.000000,NaN,NaN,medium,"[""TCGA-AR-A1AR-01A"", ""TCGA-AR-A1AR-10A"", ""TCGA...",[],[fill in during biospecimen identifier crosswa...
7,sample_identifier_like,overlapping_link_candidate,biospecimen_slide,bcr_sample_barcode,2,sample,barcode,field_name_level_and_identifier_token_match,0.0,1995,...,bcr_slide_barcode,barcode_prefix_same_row,1995,0.991479,NaN,NaN,medium,"[""TCGA-AR-A1AR-01A"", ""TCGA-BH-A1EO-01A"", ""TCGA...","[{""prefix_value"": ""TCGA-BH-A1ET-11B"", ""child_v...",[fill in during biospecimen identifier crosswa...


,identifier_linkage_family,crosswalk_role,table_name,field_name,source_position,probable_identifier_level,identifier_token_family,candidate_source,missing_like_fraction,non_missing_count,...,paired_field_name,pattern_rule,pattern_comparable_row_count,pattern_pass_fraction,alternate_column_name,cde_id_raw,manual_review_priority,example_values_small_sample,pattern_nonmatch_examples_json,notes_placeholder
0,multi_level_or_unclear_identifier_like,ambiguous_candidate,biospecimen_aliquot,biospecimen_barcode_bottom,5,multi_level_or_unclear,barcode,field_name_identifier_token_match_only,0.000000,14538,...,bcr_sample_barcode,same_row_copresence_only,14538,1.0,NaN,NaN,high,"[""0099016644"", ""0108477580"", ""0108477484"", ""01...",[],[fill in during biospecimen identifier crosswa...
1,multi_level_or_unclear_identifier_like,ambiguous_candidate,biospecimen_sample,pathology_report_uuid,18,multi_level_or_unclear,uuid,field_name_identifier_token_match_only,0.513032,1121,...,bcr_sample_barcode,same_row_copresence_only,1121,1.0,NaN,NaN,high,"[""747FB91B-F523-4FA0-91DD-6014EF55643D"", ""A2B7...",[],[fill in during biospecimen identifier crosswa...
2,multi_level_or_unclear_identifier_like,ambiguous_candidate,biospecimen_shipment_portion,shipment_portion_bcr_aliquot_barcode,3,multi_level_or_unclear,barcode,field_name_multi_level_token_match,0.000000,1064,...,bcr_sample_barcode,same_row_copresence_only,1064,1.0,NaN,NaN,high,"[""TCGA-BH-A1EO-01A-21-A17I-20"", ""TCGA-BH-A1ES-...",[],[fill in during biospecimen identifier crosswa...


## Save likely linkage chain review table


In [5]:
step_labels = {
    'patient_identifier_like': 'patient',
    'sample_identifier_like': 'sample',
    'portion_identifier_like': 'portion',
    'analyte_identifier_like': 'analyte',
    'slide_identifier_like': 'slide',
    'aliquot_identifier_like': 'aliquot',
    'multi_level_or_unclear_identifier_like': 'multi_level_or_unclear',
}

primary_chain_df = (
    crosswalk_df.loc[crosswalk_df['crosswalk_role'] == 'primary_link_candidate', [
        'identifier_linkage_family',
        'table_name',
        'field_name',
        'paired_field_name',
        'pattern_rule',
        'pattern_pass_fraction',
        'crosswalk_role',
        'notes_placeholder',
    ]]
    .assign(
        chain_observation_type='central_chain_candidate',
        linkage_step=lambda df: df['identifier_linkage_family'].map(step_labels),
        evidence_rule=lambda df: df['pattern_rule'],
        evidence_rate=lambda df: df['pattern_pass_fraction'],
        chain_comment='Primary crosswalk candidate carried forward as a central chain review observation.',
    )
)

side_link_df = (
    crosswalk_df.loc[crosswalk_df['crosswalk_role'] == 'side_table_candidate', [
        'identifier_linkage_family',
        'table_name',
        'field_name',
        'paired_field_name',
        'pattern_rule',
        'pattern_pass_fraction',
        'crosswalk_role',
        'notes_placeholder',
    ]]
    .assign(
        chain_observation_type='side_link_observation',
        linkage_step=lambda df: df['identifier_linkage_family'].map(step_labels),
        evidence_rule=lambda df: df['pattern_rule'],
        evidence_rate=lambda df: df['pattern_pass_fraction'],
        chain_comment='Side-table crosswalk candidate carried forward as a linkage-support observation only.',
    )
)

chain_df = pd.concat([primary_chain_df, side_link_df], ignore_index=True)
chain_df['identifier_linkage_family'] = pd.Categorical(
    chain_df['identifier_linkage_family'], categories=family_order, ordered=True
)
chain_df['chain_observation_type'] = pd.Categorical(
    chain_df['chain_observation_type'],
    categories=['central_chain_candidate', 'side_link_observation'],
    ordered=True,
)
chain_df = (
    chain_df[
        [
            'chain_observation_type',
            'linkage_step',
            'identifier_linkage_family',
            'table_name',
            'field_name',
            'paired_field_name',
            'evidence_rule',
            'evidence_rate',
            'crosswalk_role',
            'chain_comment',
            'notes_placeholder',
        ]
    ]
    .sort_values(
        ['chain_observation_type', 'identifier_linkage_family', 'table_name', 'field_name'],
        ascending=[True, True, True, True],
    )
    .reset_index(drop=True)
)
chain_path = results_root / '47_biospecimen_identifier_likely_linkage_chain.tsv'
chain_df.to_csv(chain_path, sep='\t', index=False)

print(f'Saved: {chain_path}')
display(chain_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\47_biospecimen_identifier_likely_linkage_chain.tsv


,chain_observation_type,linkage_step,identifier_linkage_family,table_name,field_name,paired_field_name,evidence_rule,evidence_rate,crosswalk_role,chain_comment,notes_placeholder
0,central_chain_candidate,patient,patient_identifier_like,biospecimen_diagnostic_slides,bcr_patient_barcode,bcr_patient_uuid,exact_pair_one_to_one_same_table,1.000000,primary_link_candidate,Primary crosswalk candidate carried forward as...,[fill in during biospecimen identifier crosswa...
1,central_chain_candidate,patient,patient_identifier_like,biospecimen_sample,bcr_patient_uuid,bcr_sample_barcode,same_row_copresence_only,1.000000,primary_link_candidate,Primary crosswalk candidate carried forward as...,[fill in during biospecimen identifier crosswa...
2,central_chain_candidate,sample,sample_identifier_like,biospecimen_sample,bcr_sample_barcode,bcr_sample_uuid,exact_pair_one_to_one_same_table,1.000000,primary_link_candidate,Primary crosswalk candidate carried forward as...,[fill in during biospecimen identifier crosswa...
3,central_chain_candidate,sample,sample_identifier_like,biospecimen_sample,bcr_sample_uuid,bcr_sample_barcode,exact_pair_one_to_one_same_table,1.000000,primary_link_candidate,Primary crosswalk candidate carried forward as...,[fill in during biospecimen identifier crosswa...
4,central_chain_candidate,portion,portion_identifier_like,biospecimen_portion,bcr_portion_barcode,bcr_portion_uuid,exact_pair_one_to_one_same_table,1.000000,primary_link_candidate,Primary crosswalk candidate carried forward as...,[fill in during biospecimen identifier crosswa...
5,central_chain_candidate,portion,portion_identifier_like,biospecimen_portion,bcr_portion_uuid,bcr_portion_barcode,exact_pair_one_to_one_same_table,1.000000,primary_link_candidate,Primary crosswalk candidate carried forward as...,[fill in during biospecimen identifier crosswa...
6,central_chain_candidate,analyte,analyte_identifier_like,biospecimen_analyte,bcr_analyte_barcode,bcr_analyte_uuid,exact_pair_one_to_one_same_table,1.000000,primary_link_candidate,Primary crosswalk candidate carried forward as...,[fill in during biospecimen identifier crosswa...
7,central_chain_candidate,analyte,analyte_identifier_like,biospecimen_analyte,bcr_analyte_uuid,bcr_analyte_barcode,exact_pair_one_to_one_same_table,1.000000,primary_link_candidate,Primary crosswalk candidate carried forward as...,[fill in during biospecimen identifier crosswa...
8,central_chain_candidate,slide,slide_identifier_like,biospecimen_slide,bcr_slide_barcode,bcr_slide_uuid,exact_pair_one_to_one_same_table,1.000000,primary_link_candidate,Primary crosswalk candidate carried forward as...,[fill in during biospecimen identifier crosswa...
9,central_chain_candidate,slide,slide_identifier_like,biospecimen_slide,bcr_slide_uuid,bcr_slide_barcode,exact_pair_one_to_one_same_table,1.000000,primary_link_candidate,Primary crosswalk candidate carried forward as...,[fill in during biospecimen identifier crosswa...


## Review reminder

These outputs stay at the source-audit and linkage-preparation layer. They do not define a final cohort, freeze a final endpoint, parse XML or SSF, or create a harmonized modeling-ready biospecimen hierarchy.
